In [2]:
import pandas as pd
import numpy as np
from typing import Dict, Any

# 비교에 사용할 모든 장기 수익률 컬럼
RETURN_COLUMNS_SUMMARY = ['return_post_1d', 'return_post_2d', 'return_post_5d', 'return_post_10d', 'return_post_20d']

def compile_model_comparison(model_results: Dict[str, Dict[str, Any]]) -> pd.DataFrame:
    """
    모델 버전별 최종 성능 지표를 취합하여 비교 테이블을 생성합니다.

    Args:
        model_results: {'ModelName': {'summary_df': df_avg_returns, 'precision_1d': float, 'total_signals': int}} 형태의 딕셔너리.
    """
    
    final_comparison = []
    
    for model_name, results in model_results.items():
        df_avg_return = results['summary_df']
        total_signals = results.get('total_signals', np.nan)
        precision_1d = results.get('precision_1d', np.nan)
        
        if 'BUY' not in df_avg_return.index:
            continue
            
        buy_performance = df_avg_return.loc['BUY'].copy()
        
        # 1. 기본 정보 및 정밀도
        data = {'Model': model_name, 'Total Signals': total_signals, 'Precision 1D (%)': precision_1d}
        
        # 2. 장기 수익률 컬럼 추가
        for days in [1, 2, 5, 10, 20]:
            # 예: Avg_Return_Post_1D (%)
            avg_col_name = f"Avg_Return_Post_{days}D (%)"
            data[avg_col_name] = buy_performance.get(avg_col_name, np.nan)
        
        final_comparison.append(data)

    df_comparison = pd.DataFrame(final_comparison).set_index('Model')
    
    # 컬럼 순서 조정
    order = ['Total Signals', 'Precision 1D (%)'] + [col for col in df_comparison.columns if col not in ['Total Signals', 'Precision 1D (%)']]
    
    return df_comparison[order].round(2)


# --- 예시 데이터 설정 (실제로는 각 모델의 검증 결과를 여기에 입력해야 합니다) ---

# NOTE: 이 데이터는 이전 분석 결과를 기반으로 한 가정이며, 실제 파일 로드 후 정확한 값을 입력해야 합니다.

model_v1_results = {
    'summary_df': pd.DataFrame({'Avg_Return_Post_1D (%)': [0.27], 'Avg_Return_Post_2D (%)': [0.30]}, index=['BUY']),
    'precision_1d': 49.09,
    'total_signals': 328
}

model_v4_results = {
    'summary_df': pd.DataFrame({
        'Avg_Return_Post_1D (%)': [2.50], 
        'Avg_Return_Post_2D (%)': [3.50],
        'Avg_Return_Post_5D (%)': [6.00],
        'Avg_Return_Post_10D (%)': [10.00],
        'Avg_Return_Post_20D (%)': [15.00],
    }, index=['BUY']),
    'precision_1d': 85.00, # V3/V4는 필터 추가로 Precision이 높아야 함 (가정)
    'total_signals': 10 # V3/V4는 필터 추가로 시그널 수가 매우 적어야 함 (가정)
}

model_performance_data = {
    'V1 Basic (Z-Only)': model_v1_results,
    'V4 Final Hybrid (5-Factor)': model_v4_results
}


# --- 모델 통합 및 비교 실행 ---
df_final_comparison_table = compile_model_comparison(model_performance_data)


# --- 결과 출력 ---
print("="*100)
print("FINAL MODEL PERFORMANCE COMPARISON (BUY SIGNAL)")
print("="*100)
print(df_final_comparison_table.to_markdown())
print("="*100)

FINAL MODEL PERFORMANCE COMPARISON (BUY SIGNAL)
| Model                      |   Total Signals |   Precision 1D (%) |   Avg_Return_Post_1D (%) |   Avg_Return_Post_2D (%) |   Avg_Return_Post_5D (%) |   Avg_Return_Post_10D (%) |   Avg_Return_Post_20D (%) |
|:---------------------------|----------------:|-------------------:|-------------------------:|-------------------------:|-------------------------:|--------------------------:|--------------------------:|
| V1 Basic (Z-Only)          |             328 |              49.09 |                     0.27 |                      0.3 |                      nan |                       nan |                       nan |
| V4 Final Hybrid (5-Factor) |              10 |              85    |                     2.5  |                      3.5 |                        6 |                        10 |                        15 |


In [1]:
import pandas as pd
import numpy as np
import os
from typing import Dict, Any, List

# 비교에 사용할 모든 장기 수익률 컬럼
RETURN_COLS = ['return_post_1d', 'return_post_2d', 'return_post_5d', 'return_post_10d', 'return_post_20d']
RETURN_COLS_SUMMARY = [f"Avg_Return_Post_{d}D (%)" for d in [1, 2, 5, 10, 20]]

# --- 1. 성능 비교 테이블 생성 함수 ---

def compile_model_comparison(model_results: Dict[str, Dict[str, Any]]) -> pd.DataFrame:
    """ 모델 버전별 최종 성능 지표를 취합하여 비교 테이블을 생성합니다. """
    
    final_comparison = []
    
    for model_name, results in model_results.items():
        if 'BUY' not in results:
            continue
            
        buy_performance = results['BUY']
        
        # 1. 기본 정보 및 정밀도
        data = {
            'Model Version': model_name,
            'Total Signals': buy_performance['Total Signals'],
            'Precision 1D (%)': buy_performance['Precision (%)']
        }
        
        # 2. 장기 수익률 컬럼 추가
        for i, days in enumerate([1, 2, 5, 10, 20]):
            avg_col_name = f"Avg_Return_Post_{days}D (%)"
            data[avg_col_name] = buy_performance.get(avg_col_name, np.nan)
        
        final_comparison.append(data)

    df_comparison = pd.DataFrame(final_comparison).set_index('Model Version')
    
    # 컬럼 순서 조정
    order = ['Total Signals', 'Precision 1D (%)'] + RETURN_COLS_SUMMARY
    return df_comparison[order].round(2)


# --- 2. 모델별 성능 평가 로직 (V1과 V5의 필터링 규칙 적용) ---

# V5 최종 규칙 하이퍼파라미터
V5_RULES = {
    'z_threshold': 2.0,
    'gics_sectors': [35.0],
    'momentum_threshold': 0.08,
    'volume_threshold': 2.0,
    'rsi_range': (50.0, 75.0)
}

def evaluate_model_performance(df_val_data: pd.DataFrame, rules: Dict[str, Any], model_type: str) -> Dict[str, Any]:
    """주어진 데이터와 모델 타입에 따라 성능 지표를 계산합니다."""
    
    df = df_val_data.copy()
    
    # 1. 시그널 필터링 조건 정의
    is_surprise_ok = (df['surprise_z'] > rules['z_threshold'])
    is_gics_ok = (df['gics_code'].isin(rules['gics_sectors']))
    is_momentum_ok = (df['momentum_rate'] >= rules['momentum_threshold'])
    is_volume_ok = (df['volume_ratio'] >= rules['volume_threshold'])
    is_rsi_ok = (df['rsi'] > rules['rsi_range'][0]) & (df['rsi'] <= rules['rsi_range'][1])

    if model_type == 'V1 Basic':
        # V1: Z-Score만 사용
        is_final_signal = is_surprise_ok
    else:
        # V5: 5가지 조건 모두 충족 (GICS, Momentum, Volume, RSI 추가)
        is_final_signal = is_surprise_ok & is_gics_ok & is_momentum_ok & is_volume_ok & is_rsi_ok

    # 최종 BUY 시그널 필터링
    df_signals = df[is_final_signal].copy()

    if df_signals.empty:
        return {}

    # 2. 성능 지표 계산
    
    # Precision: return_post_1d > 0
    df_signals['is_correct'] = df_signals['return_post_1d'] > 0
    
    total_signals = len(df_signals)
    precision_1d = df_signals['is_correct'].mean() * 100
    
    # 장기 수익률 계산
    df_avg_return = df_signals[[col for col in RETURN_COLS_SUMMARY if col in df_signals.columns]].mean()
    avg_return_dict = {f"Avg_Return_Post_{d}D (%)": df_avg_return[f'return_post_{d}d'] for d in [1, 2, 5, 10, 20]}

    return {'BUY': {'Total Signals': total_signals, 'Precision (%)': precision_1d, **avg_return_dict}}


# --- 3. 모델 통합 및 비교 실행 함수 ---

def execute_comparison_from_data(full_validation_data_path: str):
    
    try:
        # NOTE: 모든 필요한 컬럼을 포함하여 로드
        required_cols_load = ['symbol', 'date', 'surprise_z', 'gics_code', 'momentum_rate', 'volume_ratio', 'rsi'] + [col.replace('Avg_Return_', 'return_post_').replace('D (%)', 'd') for col in RETURN_COLS_SUMMARY]
        
        df_val_data = pd.read_csv(
            full_validation_data_path, 
            usecols=required_cols_load,
            # 모든 컬럼을 float32로 로드하여 메모리 효율성 유지
            dtype={c: np.float32 for c in required_cols_load if c not in ['symbol', 'date']}
        )
        
    except Exception as e:
        print(f"❌ 오류: 2024년 검증 데이터 로드 실패. 필요한 컬럼이 모두 있는지 확인하세요: {e}")
        return

    # V1 규칙 정의 (Z-Score만 사용, GICS/Tech 필터는 무시)
    V1_SIMPLIFIED_RULES = V5_RULES.copy()
    V1_SIMPLIFIED_RULES['gics_sectors'] = [35.0, 10.0, 20.0, 45.0] # 모든 주요 섹터를 포함하는 것으로 간주
    V1_SIMPLIFIED_RULES['momentum_threshold'] = 0.0
    V1_SIMPLIFIED_RULES['volume_threshold'] = 0.0
    V1_SIMPLIFIED_RULES['rsi_range'] = (0.0, 100.0)

    # 모델별 성능 평가 실행
    v1_performance = evaluate_model_performance(df_val_data, V1_SIMPLIFIED_RULES, 'V1 Basic')
    v5_performance = evaluate_model_performance(df_val_data, V5_RULES, 'V5 Final Hybrid')

    model_performance_data = {
        'V1 Basic (Z-Only)': v1_performance,
        'V5 Final Hybrid (5-Factor)': v5_performance
    }
    
    # 최종 비교 테이블 생성
    df_comparison = compile_model_comparison(model_performance_data)

    # --- 결과 출력 ---
    print("\n" + "="*100)
    print("FINAL MODEL PERFORMANCE COMPARISON (2024 VALIDATION)")
    print("="*100)
    print("V1은 모든 종목을 대상으로 Z>2를 적용한 기본 모델의 성능을 모사합니다.")
    print(df_comparison.to_markdown())
    print("="*100)
    print("✅ 해석:")
    print("V5 모델은 5가지 필터 적용 후 시그널 수를 극단적으로 줄이면서도, Precision과 장기 수익률을 V1 대비 대폭 향상시켜야 성공적인 전략입니다.")
    print("V1의 낮은 정밀도(49%)는 필터링의 필요성을 입증합니다.")
    print("="*100)


# --- 실행: 'test_model_2.ipynb'에 맞춘 데이터 경로 설정 ---
# NOTE: 이 코드는 VENDOR_ANALYSIS_PATH에 모든 필요한 컬럼(gics_code, momentum_rate, volume_ratio, rsi)이 통합된 파일이 저장되어 있다고 가정합니다.
FULL_VALIDATION_DATA_PATH = "../../output/problem2_vendor/problem2_vendor_analysis_base.csv"
execute_comparison_from_data(FULL_VALIDATION_DATA_PATH)

❌ 오류: 2024년 검증 데이터 로드 실패. 필요한 컬럼이 모두 있는지 확인하세요: [Errno 2] No such file or directory: '../../output/problem2_vendor/problem2_vendor_analysis_base.csv'
